# Q29 — Titanic Survival with Naive Bayes and Feature Comparison
**Topic:** Naive Bayes | Categorical Features | Feature Comparison

---
A maritime museum analytics team builds an interactive exhibit using Gaussian Naive Bayes
to explain Titanic survival patterns, then compares it with KNN.

---

In [ ]:
# ── Install / Import all libraries ──────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

# Pretty display helper
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 20)

print('All libraries imported successfully ✓')

---
## Task 114 — Load & Preprocess the Titanic Dataset

In [ ]:
# ── Load Titanic dataset (seaborn built-in, 891 rows) ────────────────────────
df = sns.load_dataset('titanic')

print('=== Raw Dataset ===')
print(f'Shape: {df.shape}')
print('\nColumn names:', df.columns.tolist())
print('\nMissing values before preprocessing:')
print(df.isnull().sum()[df.isnull().sum() > 0])

df.head()

In [ ]:
# ── Keep only the 7 original Titanic features + target ──────────────────────
# seaborn's titanic has redundant derived columns; use the base ones
KEEP = ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
df = df[KEEP].copy()

# Step 1 – Impute 'age' with MEDIAN
age_median = df['age'].median()
df['age'] = df['age'].fillna(age_median)
print(f"Age median used for imputation: {age_median}")

# Step 2 – Drop 'Cabin' (already not in our selection; confirm)
print("'cabin' column not present in selected features — dropped by selection ✓")

# Step 3 – Impute 'embarked' with MODE
embarked_mode = df['embarked'].mode()[0]
df['embarked'] = df['embarked'].fillna(embarked_mode)
print(f"Embarked mode used for imputation: '{embarked_mode}'")

# Step 4 – Encode 'sex': male=0, female=1
df['sex'] = df['sex'].map({'male': 0, 'female': 1})

# Step 5 – One-hot encode 'embarked'
embarked_dummies = pd.get_dummies(df['embarked'], prefix='embarked', drop_first=False).astype(int)
df = pd.concat([df.drop(columns='embarked'), embarked_dummies], axis=1)

print(f'\n=== After Preprocessing ===')
print(f'Shape: {df.shape}')
print('Missing values:', df.isnull().sum().sum())
print('\nFinal Feature List:')
features = [c for c in df.columns if c != 'survived']
for i, f in enumerate(features, 1):
    print(f'  {i}. {f}')

df.head()

---
## Task 115 — Train Gaussian Naive Bayes (80/20 Stratified Split)

In [ ]:
# ── Features & Target ────────────────────────────────────────────────────────
X = df[features]
y = df['survived']

# ── 80/20 Stratified Split ───────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Training set size : {X_train.shape[0]} samples')
print(f'Test set size     : {X_test.shape[0]} samples')
print(f'Survival rate (train): {y_train.mean():.4f}')
print(f'Survival rate (test) : {y_test.mean():.4f}')

# ── Train Gaussian Naive Bayes ───────────────────────────────────────────────
gnb = GaussianNB()
gnb.fit(X_train, y_train)

y_train_pred = gnb.predict(X_train)
y_test_pred  = gnb.predict(X_test)

train_acc = accuracy_score(y_train, y_train_pred)
test_acc  = accuracy_score(y_test,  y_test_pred)

print(f'\n=== Gaussian Naive Bayes Accuracy ===')
print(f'Training Accuracy : {train_acc:.4f}  ({train_acc*100:.2f}%)')
print(f'Test Accuracy     : {test_acc:.4f}  ({test_acc*100:.2f}%)')

---
## Task 116 — Precision, Recall, F1, MCC per Class

In [ ]:
# ── Per-class metrics ────────────────────────────────────────────────────────
labels = [0, 1]
label_names = ['Not Survived (0)', 'Survived (1)']

prec   = precision_score(y_test, y_test_pred, labels=labels, average=None)
rec    = recall_score   (y_test, y_test_pred, labels=labels, average=None)
f1     = f1_score       (y_test, y_test_pred, labels=labels, average=None)
mcc    = matthews_corrcoef(y_test, y_test_pred)   # MCC is a single scalar

metrics_df = pd.DataFrame({
    'Class'    : label_names,
    'Precision': prec,
    'Recall'   : rec,
    'F1-Score' : f1,
    'MCC (overall)': [mcc, mcc]
})

print('=== GNB Metrics on Test Set ===')
print(metrics_df.to_string(index=False))
print(f'\nOverall MCC = {mcc:.4f}')

# ── Visualise Confusion Matrix ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cm = confusion_matrix(y_test, y_test_pred)
ConfusionMatrixDisplay(cm, display_labels=['Not Survived','Survived']).plot(
    ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('GNB Confusion Matrix', fontsize=13, fontweight='bold')

# Bar chart of metrics per class
x = np.arange(len(label_names))
w = 0.25
axes[1].bar(x - w, prec, w, label='Precision', color='steelblue')
axes[1].bar(x,     rec,  w, label='Recall',    color='darkorange')
axes[1].bar(x + w, f1,   w, label='F1-Score',  color='seagreen')
axes[1].set_xticks(x)
axes[1].set_xticklabels(label_names)
axes[1].set_ylim(0, 1.1)
axes[1].set_ylabel('Score')
axes[1].set_title('Precision / Recall / F1 per Class (GNB)', fontweight='bold')
axes[1].legend()
axes[1].axhline(mcc, color='red', linestyle='--', label=f'MCC={mcc:.3f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('/home/claude/fig_task116_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved ✓')

---
## Task 117 — Class-Conditional Mean & Variance (Age and Fare)

In [ ]:
# GNB stores theta_ (mean) and var_ (variance) per class per feature
feature_list = X_train.columns.tolist()
age_idx  = feature_list.index('age')
fare_idx = feature_list.index('fare')

classes = gnb.classes_   # [0, 1]

stats_data = []
for ci, cls in enumerate(classes):
    stats_data.append({
        'Class'       : 'Not Survived' if cls == 0 else 'Survived',
        'Age Mean'    : gnb.theta_[ci][age_idx],
        'Age Variance': gnb.var_[ci][age_idx],
        'Fare Mean'   : gnb.theta_[ci][fare_idx],
        'Fare Variance': gnb.var_[ci][fare_idx],
    })

stats_df = pd.DataFrame(stats_data)
print('=== Class-Conditional Statistics Learned by GNB ===')
print(stats_df.to_string(index=False))

fare_surv     = stats_df[stats_df['Class']=='Survived']['Fare Mean'].values[0]
fare_non_surv = stats_df[stats_df['Class']=='Not Survived']['Fare Mean'].values[0]
ans = 'YES' if fare_surv > fare_non_surv else 'NO'
print(f'\nDo survivors have a higher mean Fare? → {ans}')
print(f'  Survivors mean Fare     = {fare_surv:.4f}')
print(f'  Non-survivors mean Fare = {fare_non_surv:.4f}')

# ── Visualisation ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for feature, ax in zip(['age', 'fare'], axes):
    for label, color in zip([0, 1], ['tomato', 'steelblue']):
        subset = df[df['survived'] == label][feature]
        ax.hist(subset, bins=30, alpha=0.6,
                color=color,
                label='Survived' if label == 1 else 'Not Survived',
                density=True)
        mu  = gnb.theta_[label][feature_list.index(feature)]
        std = np.sqrt(gnb.var_[label][feature_list.index(feature)])
        x   = np.linspace(subset.min(), subset.max(), 200)
        ax.plot(x, stats.norm.pdf(x, mu, std), linewidth=2.5,
                color=('darkred' if label==0 else 'navy'),
                linestyle='--', label=f'GNB Gaussian μ={mu:.1f}')
    ax.set_xlabel(feature.capitalize(), fontsize=12)
    ax.set_ylabel('Density', fontsize=12)
    ax.set_title(f'Class-Conditional Distribution: {feature.capitalize()}',
                 fontweight='bold', fontsize=13)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('/home/claude/fig_task117_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved ✓')

---
## Task 118 — 10-Fold Cross-Validation

In [ ]:
# ── 10-fold Stratified CV on full dataset ───────────────────────────────────
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
cv_scores = cross_val_score(GaussianNB(), X, y, cv=skf, scoring='accuracy')

cv_mean = cv_scores.mean()
cv_std  = cv_scores.std()

print('=== 10-Fold Cross-Validation Results (GNB) ===')
print(f'Fold Accuracies : {[round(s,4) for s in cv_scores]}')
print(f'Mean Accuracy   : {cv_mean:.4f}  ({cv_mean*100:.2f}%)')
print(f'Std Dev         : {cv_std:.4f}')
print(f'\nIs variance low (std < 0.05)? → {"YES" if cv_std < 0.05 else "NO"} '
      f'(std = {cv_std:.4f})')

# ── Visualise CV scores ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(1, 11), cv_scores, color='steelblue', alpha=0.8, edgecolor='black')
ax.axhline(cv_mean, color='red', linestyle='--', linewidth=2,
           label=f'Mean = {cv_mean:.4f}')
ax.fill_between(range(1, 11),
                cv_mean - cv_std, cv_mean + cv_std,
                alpha=0.15, color='red', label=f'±1 std ({cv_std:.4f})')
ax.set_xticks(range(1, 11))
ax.set_xlabel('Fold', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('10-Fold Cross-Validation Accuracy (Gaussian Naive Bayes)',
             fontweight='bold', fontsize=13)
ax.set_ylim(0.6, 1.0)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('/home/claude/fig_task118_cv.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved ✓')

---
## Task 119 — Train KNN (K=7) and Compare with GNB

In [ ]:
# ── Scale features for KNN ───────────────────────────────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# ── Train KNN K=7 ────────────────────────────────────────────────────────────
knn = KNeighborsClassifier(n_neighbors=7)
knn.fit(X_train_scaled, y_train)
y_knn_pred = knn.predict(X_test_scaled)

# ── Collect metrics for both models ─────────────────────────────────────────
def get_metrics(y_true, y_pred, model_name):
    prec_arr = precision_score(y_true, y_pred, labels=[0,1], average=None)
    rec_arr  = recall_score   (y_true, y_pred, labels=[0,1], average=None)
    f1_arr   = f1_score       (y_true, y_pred, labels=[0,1], average=None)
    acc      = accuracy_score (y_true, y_pred)
    mcc_val  = matthews_corrcoef(y_true, y_pred)
    rows = []
    for i, cls in enumerate(['Not Survived (0)', 'Survived (1)']):
        rows.append({
            'Model'    : model_name,
            'Class'    : cls,
            'Accuracy' : round(acc, 4),
            'Precision': round(prec_arr[i], 4),
            'Recall'   : round(rec_arr[i], 4),
            'F1-Score' : round(f1_arr[i], 4),
            'MCC'      : round(mcc_val, 4)
        })
    return rows

gnb_rows = get_metrics(y_test, y_test_pred, 'Gaussian NB')
knn_rows = get_metrics(y_test, y_knn_pred,  'KNN (K=7)')

compare_df = pd.DataFrame(gnb_rows + knn_rows)
print('=== Model Comparison: GNB vs KNN (K=7) ===')
print(compare_df.to_string(index=False))

gnb_f1_surv = compare_df[(compare_df['Model']=='Gaussian NB') &
                          (compare_df['Class']=='Survived (1)')]['F1-Score'].values[0]
knn_f1_surv = compare_df[(compare_df['Model']=='KNN (K=7)') &
                          (compare_df['Class']=='Survived (1)')]['F1-Score'].values[0]
winner = 'KNN (K=7)' if knn_f1_surv > gnb_f1_surv else 'Gaussian NB'
print(f'\nWhich model achieves higher F1 for Survivors?')
print(f'  GNB F1 (Survived) = {gnb_f1_surv}   KNN F1 (Survived) = {knn_f1_surv}')
print(f'  → {winner} achieves higher F1 for Survivors')

In [ ]:
# ── Side-by-side visual comparison ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metric_cols = ['Precision', 'Recall', 'F1-Score']
colors_map = {'Gaussian NB': 'steelblue', 'KNN (K=7)': 'darkorange'}

for ax, cls_name in zip(axes, ['Not Survived (0)', 'Survived (1)']):
    subset = compare_df[compare_df['Class'] == cls_name]
    x = np.arange(len(metric_cols))
    w = 0.35
    for j, (_, row) in enumerate(subset.iterrows()):
        offset = (j - 0.5) * w
        bars = ax.bar(x + offset, [row[m] for m in metric_cols],
                      width=w, label=row['Model'],
                      color=colors_map[row['Model']], alpha=0.85,
                      edgecolor='black')
        for bar in bars:
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 0.01,
                    f'{bar.get_height():.3f}',
                    ha='center', va='bottom', fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels(metric_cols, fontsize=11)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel('Score')
    ax.set_title(f'GNB vs KNN — Class: {cls_name}',
                 fontweight='bold', fontsize=12)
    ax.legend()

plt.suptitle('Task 119 — GNB vs KNN (K=7) Per-Class Metrics',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/home/claude/fig_task119_gnb_vs_knn.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved ✓')

---
## Task 120 — Independence Assumption Check: Pearson Correlation (Fare vs Pclass)

In [ ]:
# ── Pearson Correlation ──────────────────────────────────────────────────────
corr_val, p_value = stats.pearsonr(df['fare'], df['pclass'])

print('=== Pearson Correlation: Fare vs Pclass ===')
print(f'Correlation coefficient (r) = {corr_val:.4f}')
print(f'P-value                     = {p_value:.2e}')
print()
print(f'Is |r| > 0.5?  → {"YES" if abs(corr_val) > 0.5 else "NO"} (|r| = {abs(corr_val):.4f})')
print()
print('Interpretation:')
if abs(corr_val) > 0.5:
    print('  Strong correlation exists between Fare and Pclass.')
    print('  This VIOLATES the Naive Bayes independence assumption')
    print('  (these two features are NOT independent given the class).')
else:
    print('  Weak correlation; independence assumption is roughly satisfied.')

# ── Scatter + Regression plot ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter
jitter = np.random.RandomState(0).uniform(-0.1, 0.1, len(df))
axes[0].scatter(df['pclass'] + jitter, df['fare'],
                alpha=0.4, c='steelblue', s=20)
m, b = np.polyfit(df['pclass'], df['fare'], 1)
x_line = np.array([1, 2, 3])
axes[0].plot(x_line, m*x_line + b, 'r--', linewidth=2,
             label=f'Regression line (r={corr_val:.3f})')
axes[0].set_xlabel('Passenger Class (Pclass)', fontsize=12)
axes[0].set_ylabel('Fare', fontsize=12)
axes[0].set_title('Fare vs Pclass (Independence Check)',
                  fontweight='bold', fontsize=13)
axes[0].set_xticks([1, 2, 3])
axes[0].set_xticklabels(['1st Class', '2nd Class', '3rd Class'])
axes[0].legend()

# Box plot by class
for i, cls in enumerate([1, 2, 3]):
    axes[1].boxplot(df[df['pclass']==cls]['fare'], positions=[cls],
                    widths=0.5, patch_artist=True,
                    boxprops=dict(facecolor=['lightblue','lightgreen','salmon'][i], alpha=0.7))
axes[1].set_xticks([1, 2, 3])
axes[1].set_xticklabels(['1st Class', '2nd Class', '3rd Class'])
axes[1].set_xlabel('Passenger Class', fontsize=12)
axes[1].set_ylabel('Fare', fontsize=12)
axes[1].set_title('Fare Distribution by Pclass (Box Plot)',
                  fontweight='bold', fontsize=13)

plt.suptitle(f'Task 120 — Pearson r(Fare, Pclass) = {corr_val:.4f}',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/home/claude/fig_task120_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved ✓')

---
## Task 121 — Compare GNB vs KNN Using MCC

In [ ]:
# ── MCC Comparison ───────────────────────────────────────────────────────────
gnb_acc  = accuracy_score(y_test, y_test_pred)
knn_acc  = accuracy_score(y_test, y_knn_pred)
gnb_mcc  = matthews_corrcoef(y_test, y_test_pred)
knn_mcc  = matthews_corrcoef(y_test, y_knn_pred)

summary_df = pd.DataFrame({
    'Model'   : ['Gaussian NB', 'KNN (K=7)'],
    'Accuracy': [gnb_acc, knn_acc],
    'MCC'     : [gnb_mcc, knn_mcc]
})

print('=== Task 121 — GNB vs KNN: Accuracy & MCC ===')
print(summary_df.to_string(index=False))

higher_acc = 'GNB' if gnb_acc > knn_acc else 'KNN'
higher_mcc = 'GNB' if gnb_mcc > knn_mcc else 'KNN'
print(f'\nModel with higher Accuracy : {higher_acc}')
print(f'Model with higher MCC      : {higher_mcc}')
if higher_acc != higher_mcc:
    print('→ The model with higher Accuracy does NOT necessarily have higher MCC.')
    print('  MCC is a more balanced metric — better for imbalanced datasets.')
else:
    print('→ Both metrics agree on the better model in this case.')

# ── Summary Dashboard Plot ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

models   = ['Gaussian NB', 'KNN (K=7)']
accs     = [gnb_acc, knn_acc]
mccs     = [gnb_mcc, knn_mcc]
palette  = ['steelblue', 'darkorange']

# Accuracy bar
bars0 = axes[0].bar(models, accs, color=palette, alpha=0.85, edgecolor='black', width=0.5)
for bar, val in zip(bars0, accs):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                 f'{val:.4f}', ha='center', va='bottom', fontweight='bold')
axes[0].set_ylim(0, 1.1)
axes[0].set_title('Test Accuracy', fontweight='bold', fontsize=13)
axes[0].set_ylabel('Score')

# MCC bar
bars1 = axes[1].bar(models, mccs, color=palette, alpha=0.85, edgecolor='black', width=0.5)
for bar, val in zip(bars1, mccs):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                 f'{val:.4f}', ha='center', va='bottom', fontweight='bold')
axes[1].set_ylim(0, 1.1)
axes[1].set_title('MCC (Matthews Correlation Coefficient)',
                  fontweight='bold', fontsize=12)
axes[1].set_ylabel('MCC')

# Radar / Spider data as grouped bar — all metrics
all_metrics_gnb = [
    gnb_acc,
    precision_score(y_test, y_test_pred, average='weighted'),
    recall_score   (y_test, y_test_pred, average='weighted'),
    f1_score       (y_test, y_test_pred, average='weighted'),
    gnb_mcc
]
all_metrics_knn = [
    knn_acc,
    precision_score(y_test, y_knn_pred, average='weighted'),
    recall_score   (y_test, y_knn_pred, average='weighted'),
    f1_score       (y_test, y_knn_pred, average='weighted'),
    knn_mcc
]
metric_names = ['Accuracy', 'Precision\n(wtd)', 'Recall\n(wtd)', 'F1\n(wtd)', 'MCC']
x = np.arange(len(metric_names))
w = 0.35
axes[2].bar(x - w/2, all_metrics_gnb, w, label='Gaussian NB',
            color='steelblue', alpha=0.85, edgecolor='black')
axes[2].bar(x + w/2, all_metrics_knn, w, label='KNN (K=7)',
            color='darkorange', alpha=0.85, edgecolor='black')
axes[2].set_xticks(x)
axes[2].set_xticklabels(metric_names, fontsize=9)
axes[2].set_ylim(0, 1.15)
axes[2].set_title('Full Metric Comparison (GNB vs KNN)',
                  fontweight='bold', fontsize=12)
axes[2].set_ylabel('Score')
axes[2].legend()

plt.suptitle('Task 121 — Model Comparison Dashboard',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/home/claude/fig_task121_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved ✓')

---
## Guided Interpretation Questions

In [ ]:
print('=' * 70)
print('GUIDED INTERPRETATION — ANSWERS')
print('=' * 70)
print()
print('Q1: Is the GNB independence assumption violated by Fare–Pclass correlation?')
print(f'    Pearson r = {corr_val:.4f} → |r| > 0.5 = {abs(corr_val)>0.5}')
if abs(corr_val) > 0.5:
    print('    → YES, there is a strong negative correlation between Fare and Pclass.')
    print('      Higher-class passengers (Pclass=1) paid much higher fares.')
    print('      This violates the NB independence assumption because knowing one')
    print('      feature gives information about the other.')
print()
print('    Does this violation harm GNB relative to KNN?')
print(f'    GNB F1 (Survived) = {gnb_f1_surv}   KNN F1 (Survived) = {knn_f1_surv}')
if gnb_f1_surv < knn_f1_surv:
    print('    → KNN achieves better F1 for survivors. The independence violation')
    print('      likely contributes to GNB underperforming, but GNB is still')
    print('      competitive overall due to its robustness to mild violations.')
else:
    print('    → GNB achieves equal or better F1 despite the violation, showing')
    print('      Naive Bayes can still work well even when independence is violated.')
print()
print('Q2: Does the model with higher accuracy also have higher MCC?')
print(f'    Higher Accuracy: {higher_acc}  |  Higher MCC: {higher_mcc}')
if higher_acc == higher_mcc:
    print('    → Yes, both agree. However, accuracy alone is insufficient for')
    print('      imbalanced classes. MCC accounts for all four confusion matrix')
    print('      cells and gives a more honest picture of predictive quality.')
else:
    print('    → NO! The model with higher accuracy does NOT have higher MCC.')
    print('      For imbalanced survival labels, MCC is the more reliable metric.')
    print('      It considers true/false positives AND negatives simultaneously,')
    print('      unlike accuracy which can be misleading on skewed class distributions.')

---
## Business Translation [6 marks]

In [ ]:
business_translation = """
╔══════════════════════════════════════════════════════════════════════════╗
║            BUSINESS TRANSLATION — Titanic Survival Exhibit              ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  1. WHICH PASSENGER CHARACTERISTICS MOST STRONGLY PREDICTED SURVIVAL?   ║
║     Women survived at dramatically higher rates than men, and passengers ║
║     who paid higher ticket prices were far more likely to make it out    ║
║     alive. Younger age and smaller family size also gave passengers a    ║
║     slight edge in the chaos that unfolded that night.                   ║
║                                                                          ║
║  2. DID HIGHER-CLASS PASSENGERS FARE BETTER THAN LOWER-CLASS ONES?      ║
║     Yes — quite noticeably. First-class travellers, who occupied the     ║
║     upper decks and had quicker access to lifeboats, survived at much    ║
║     higher rates than those in third class, who were largely below deck. ║
║     Interestingly, the cabin class and ticket price were closely linked, ║
║     meaning these two facts about a passenger tell essentially the same  ║
║     story.                                                               ║
║                                                                          ║
║  3. WHAT DO THE RECORDS REVEAL ABOUT DIFFERENT GROUPS?                  ║
║     The historical records paint a clear picture: those with wealth and  ║
║     status had a significant survival advantage. Women were prioritised  ║
║     in the "women and children first" boarding of lifeboats, while men   ║
║     in lower-priced cabins faced the greatest risk. The data essentially ║
║     confirms that during the rescue, a person's social standing was a    ║
║     major factor in whether they lived or died.                          ║
║                                                                          ║
╚══════════════════════════════════════════════════════════════════════════╝
"""
print(business_translation)

---
## Final Summary Dashboard

In [ ]:
# ── Comprehensive Summary Plot ────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 14))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# 1. Survival count
ax1 = fig.add_subplot(gs[0, 0])
survival_counts = df['survived'].value_counts().sort_index()
ax1.bar(['Not Survived', 'Survived'], survival_counts.values,
        color=['tomato', 'steelblue'], edgecolor='black', alpha=0.85)
for i, v in enumerate(survival_counts.values):
    ax1.text(i, v + 5, str(v), ha='center', fontweight='bold')
ax1.set_title('Class Distribution', fontweight='bold')
ax1.set_ylabel('Count')

# 2. Survival by sex
ax2 = fig.add_subplot(gs[0, 1])
sex_surv = df.groupby(['sex', 'survived']).size().unstack()
sex_surv.index = ['Male (0)', 'Female (1)']
sex_surv.plot(kind='bar', ax=ax2, color=['tomato','steelblue'],
              edgecolor='black', alpha=0.85)
ax2.set_title('Survival by Sex', fontweight='bold')
ax2.set_ylabel('Count')
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=0)
ax2.legend(['Not Survived','Survived'])

# 3. Survival by pclass
ax3 = fig.add_subplot(gs[0, 2])
cls_surv = df.groupby(['pclass', 'survived']).size().unstack()
cls_surv.index = ['1st', '2nd', '3rd']
cls_surv.plot(kind='bar', ax=ax3, color=['tomato','steelblue'],
              edgecolor='black', alpha=0.85)
ax3.set_title('Survival by Pclass', fontweight='bold')
ax3.set_ylabel('Count')
ax3.set_xticklabels(ax3.get_xticklabels(), rotation=0)
ax3.legend(['Not Survived','Survived'])

# 4. Age distribution by survival
ax4 = fig.add_subplot(gs[1, 0])
for lbl, col in [(0,'tomato'), (1,'steelblue')]:
    ax4.hist(df[df['survived']==lbl]['age'], bins=25, alpha=0.6,
             color=col, density=True,
             label='Survived' if lbl==1 else 'Not Survived')
ax4.set_title('Age Distribution by Survival', fontweight='bold')
ax4.set_xlabel('Age'); ax4.set_ylabel('Density')
ax4.legend(fontsize=8)

# 5. Fare distribution by survival
ax5 = fig.add_subplot(gs[1, 1])
for lbl, col in [(0,'tomato'), (1,'steelblue')]:
    data_clip = np.clip(df[df['survived']==lbl]['fare'], 0, 300)
    ax5.hist(data_clip, bins=25, alpha=0.6, color=col, density=True,
             label='Survived' if lbl==1 else 'Not Survived')
ax5.set_title('Fare Distribution by Survival', fontweight='bold')
ax5.set_xlabel('Fare (capped @ 300)'); ax5.set_ylabel('Density')
ax5.legend(fontsize=8)

# 6. CV scores
ax6 = fig.add_subplot(gs[1, 2])
ax6.plot(range(1,11), cv_scores, 'o-', color='steelblue', linewidth=2, markersize=7)
ax6.axhline(cv_mean, color='red', linestyle='--', label=f'Mean={cv_mean:.3f}')
ax6.fill_between(range(1,11), cv_mean-cv_std, cv_mean+cv_std,
                 alpha=0.2, color='red')
ax6.set_title('10-Fold CV Accuracy (GNB)', fontweight='bold')
ax6.set_xlabel('Fold'); ax6.set_ylabel('Accuracy')
ax6.set_xticks(range(1,11))
ax6.legend(fontsize=8)

# 7. Correlation heatmap
ax7 = fig.add_subplot(gs[2, 0])
corr_matrix = df[['pclass','age','fare','sibsp','parch','sex']].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            ax=ax7, cbar=True, linewidths=0.5, annot_kws={'size':8})
ax7.set_title('Feature Correlation Heatmap', fontweight='bold')
ax7.tick_params(axis='x', rotation=45)

# 8. GNB vs KNN final metrics
ax8 = fig.add_subplot(gs[2, 1:])
metric_names2 = ['Accuracy', 'Prec(0)', 'Rec(0)', 'F1(0)',
                 'Prec(1)', 'Rec(1)', 'F1(1)', 'MCC']

gnb_vals = [
    gnb_acc,
    precision_score(y_test, y_test_pred, labels=[0,1], average=None)[0],
    recall_score   (y_test, y_test_pred, labels=[0,1], average=None)[0],
    f1_score       (y_test, y_test_pred, labels=[0,1], average=None)[0],
    precision_score(y_test, y_test_pred, labels=[0,1], average=None)[1],
    recall_score   (y_test, y_test_pred, labels=[0,1], average=None)[1],
    f1_score       (y_test, y_test_pred, labels=[0,1], average=None)[1],
    gnb_mcc
]
knn_vals = [
    knn_acc,
    precision_score(y_test, y_knn_pred, labels=[0,1], average=None)[0],
    recall_score   (y_test, y_knn_pred, labels=[0,1], average=None)[0],
    f1_score       (y_test, y_knn_pred, labels=[0,1], average=None)[0],
    precision_score(y_test, y_knn_pred, labels=[0,1], average=None)[1],
    recall_score   (y_test, y_knn_pred, labels=[0,1], average=None)[1],
    f1_score       (y_test, y_knn_pred, labels=[0,1], average=None)[1],
    knn_mcc
]

x = np.arange(len(metric_names2))
w = 0.35
b1 = ax8.bar(x - w/2, gnb_vals, w, label='Gaussian NB',
              color='steelblue', alpha=0.85, edgecolor='black')
b2 = ax8.bar(x + w/2, knn_vals, w, label='KNN (K=7)',
              color='darkorange', alpha=0.85, edgecolor='black')
ax8.set_xticks(x)
ax8.set_xticklabels(metric_names2, fontsize=10)
ax8.set_ylim(0, 1.2)
ax8.set_ylabel('Score')
ax8.set_title('Complete Metric Comparison — GNB vs KNN (K=7)',
              fontweight='bold', fontsize=12)
ax8.legend(fontsize=10)
for bar in list(b1) + list(b2):
    ax8.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
             f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=7)

plt.suptitle('Q29 — Titanic Survival: Complete Analysis Dashboard',
             fontsize=16, fontweight='bold', y=1.01)
plt.savefig('/home/claude/fig_final_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Final dashboard saved ✓')
print('\n=== ALL TASKS COMPLETED SUCCESSFULLY ===')